# Assignment 3 – Alzheimer's from handwriting

## Task 2: Preprocessing
### 2.1 Load data, remove ID, encode target, train/test split
- ID removed (only identifies participants)
- H = 0, P = 1 (patients = positive class → recall measures how many patients we find)
- Stratified 80/20 split, random_state=42 (same split as Assignment 1)
- The test set is NOT used until the final evaluation

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Les inn data
df = pd.read_csv("data.csv")

# 2. Fjern ID-kolonnen
df = df.drop(columns=["ID"])

# 3. Skill svaret (y) fra målingene (X), og gjør H/P om til 0/1
y = df["class"].map({"H": 0, "P": 1})
X = df.drop(columns=["class"])

# 4. Del i trening (80 %) og test (20 %)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Alle data:", X.shape)
print("Trening:", X_train.shape, "| Test:", X_test.shape)
print("Klasser i trening:", y_train.value_counts().to_dict())
print("Klasser i test:", y_test.value_counts().to_dict())


Alle data: (174, 450)
Trening: (139, 450) | Test: (35, 450)
Klasser i trening: {1: 71, 0: 68}
Klasser i test: {1: 18, 0: 17}


### 2.2 IQR capping + scaling in a Pipeline
- Capping: extreme values are clipped to Q1 − 1.5·IQR and Q3 + 1.5·IQR (no participants removed)
- StandardScaler: all features get mean 0 and std 1
- Pipeline: bounds and scaling are re-learned on the training folds in each CV round → no data leakage
- Sensitivity check (with/without capping) follows in Task 5, based on the Assignment 1 feedback

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


class IQRCapper(BaseEstimator, TransformerMixin):
    """Kutter ekstreme verdier ned til IQR-grensene (winsorizing)."""

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1 = np.percentile(X, 25, axis=0)
        q3 = np.percentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower_ = q1 - self.factor * iqr
        self.upper_ = q3 + self.factor * iqr
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower_, self.upper_)


preprocessing = Pipeline([
    ("capper", IQRCapper()),
    ("scaler", StandardScaler()),
])

# Kontroll: samme grense som i assignment 1?
X_train_prep = preprocessing.fit_transform(X_train)
col = list(X_train.columns).index("max_y_extension5")
print("Øvre grense max_y_extension5:", preprocessing.named_steps["capper"].upper_[col])
print("Gjennomsnitt etter skalering:", X_train_prep.mean().round(3))
print("Standardavvik etter skalering:", X_train_prep.std().round(3))

Øvre grense max_y_extension5: 2781.5
Gjennomsnitt etter skalering: 0.0
Standardavvik etter skalering: 1.0


## Task 3: Feature extraction
### 3.1 Aggregate each measurement across the 25 tasks
For each of the 18 measurements: mean (general level) and std (consistency across tasks) → 36 features.
Aggregation happens after standardisation, so that every task counts equally.

In [5]:
import re


class TaskAggregator(BaseEstimator, TransformerMixin):
    """Lager gjennomsnitt og standardavvik for hver måling over de 25 oppgavene."""

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        # Fjern tallet på slutten: "pressure_mean17" -> "pressure_mean"
        names = [re.sub(r"\d+$", "", c) for c in self.columns]
        self.measures_ = sorted(set(names))
        # For hver måling: hvilke kolonner (oppgaver) hører til den?
        self.groups_ = {m: [i for i, n in enumerate(names) if n == m] for m in self.measures_}
        self.feature_names_ = [f"{m}_{s}" for m in self.measures_ for s in ("mean", "std")]
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        new = []
        for m in self.measures_:
            block = X[:, self.groups_[m]]      # de 25 kolonnene for denne målingen
            new.append(block.mean(axis=1))     # gjennomsnitt per person
            new.append(block.std(axis=1))      # standardavvik per person
        return np.column_stack(new)


fe_pipeline = Pipeline([
    ("capper", IQRCapper()),
    ("scaler", StandardScaler()),
    ("aggregator", TaskAggregator(columns=list(X_train.columns))),
    ("scaler2", StandardScaler()),
])

X_train_fe = fe_pipeline.fit_transform(X_train)
agg = fe_pipeline.named_steps["aggregator"]

print("Før:", X_train.shape, "| Etter:", X_train_fe.shape)
print("Oppgaver per måling:", {m: len(i) for m, i in agg.groups_.items()})
print("Eksempler på nye features:", agg.feature_names_[:4])

Før: (139, 450) | Etter: (139, 36)
Oppgaver per måling: {'air_time': 25, 'disp_index': 25, 'gmrt_in_air': 25, 'gmrt_on_paper': 25, 'max_x_extension': 25, 'max_y_extension': 25, 'mean_acc_in_air': 25, 'mean_acc_on_paper': 25, 'mean_gmrt': 25, 'mean_jerk_in_air': 25, 'mean_jerk_on_paper': 25, 'mean_speed_in_air': 25, 'mean_speed_on_paper': 25, 'num_of_pendown': 25, 'paper_time': 25, 'pressure_mean': 25, 'pressure_var': 25, 'total_time': 25}
Eksempler på nye features: ['air_time_mean', 'air_time_std', 'disp_index_mean', 'disp_index_std']


In [6]:
fe_df = pd.DataFrame(X_train_fe, columns=agg.feature_names_, index=X_train.index)

# Forskjell i gjennomsnitt mellom pasienter (1) og friske (0)
diff = fe_df[y_train == 1].mean() - fe_df[y_train == 0].mean()
top = diff.loc[diff.abs().sort_values(ascending=False).index]

print("Features med størst forskjell mellom P og H (i standardavvik):")
print(top.head(8).round(2))

Features med størst forskjell mellom P og H (i standardavvik):
paper_time_std        1.53
total_time_mean       1.50
total_time_std        1.49
air_time_mean         1.49
air_time_std          1.48
paper_time_mean       1.38
num_of_pendown_std    1.36
disp_index_std        1.30
dtype: float64


### 3.2 Comparison: 450 original vs 36 extracted features
Same models, same 5-fold stratified cross-validation on the training set. Only the features differ.

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "KNN": KNeighborsClassifier(),
    "SVM (RBF)": SVC(kernel="rbf"),
    "Decision tree": DecisionTreeClassifier(random_state=42),
}


def make_pipeline(model, extract):
    steps = [("capper", IQRCapper()), ("scaler", StandardScaler())]
    if extract:
        steps += [
            ("aggregator", TaskAggregator(columns=list(X_train.columns))),
            ("scaler2", StandardScaler()),
        ]
    steps.append(("model", model))
    return Pipeline(steps)


rows = []
for name, model in models.items():
    for extract, label in [(False, "450 original"), (True, "36 extracted")]:
        scores = cross_validate(
            make_pipeline(model, extract), X_train, y_train,
            cv=cv, scoring=["accuracy", "recall"],
        )
        rows.append({
            "Model": name,
            "Features": label,
            "Accuracy": f"{scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}",
            "Recall (P)": f"{scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}",
        })

print(pd.DataFrame(rows).to_string(index=False))

        Model     Features      Accuracy    Recall (P)
          KNN 450 original 0.741 ± 0.067 0.490 ± 0.137
          KNN 36 extracted 0.928 ± 0.051 0.958 ± 0.034
    SVM (RBF) 450 original 0.935 ± 0.069 0.944 ± 0.053
    SVM (RBF) 36 extracted 0.914 ± 0.083 0.915 ± 0.083
Decision tree 450 original 0.813 ± 0.048 0.803 ± 0.095
Decision tree 36 extracted 0.885 ± 0.041 0.901 ± 0.073


## Task 4: Feature selection
### 4.1 Correlation filter
Many of the 36 features are almost identical (11 pairs with |r| > 0.9).
For each such pair we keep one feature and remove the other → less redundancy, same interpretable features.

In [9]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    """Fjerner features som har korrelasjon over en grense med en feature vi allerede har beholdt."""

    def __init__(self, threshold=0.9):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        corr = np.abs(np.corrcoef(X, rowvar=False))   # korrelasjon mellom alle par av kolonner
        keep = []
        for j in range(corr.shape[0]):
            # Behold kolonne j bare hvis den ikke er for lik noen vi allerede har beholdt
            if all(corr[j, k] <= self.threshold for k in keep):
                keep.append(j)
        self.keep_ = keep
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.keep_]


# Se hvilke features filteret fjerner (beregnet på treningsdata)
corr_filter = CorrelationFilter(threshold=0.9).fit(X_train_fe)
names = agg.feature_names_

removed = [names[i] for i in range(len(names)) if i not in corr_filter.keep_]
print("Før:", len(names), "| Etter:", len(corr_filter.keep_))
print("Fjernet:", removed)

Før: 36 | Etter: 28
Fjernet: ['mean_gmrt_mean', 'mean_jerk_in_air_mean', 'mean_jerk_in_air_std', 'mean_speed_in_air_mean', 'mean_speed_on_paper_mean', 'mean_speed_on_paper_std', 'total_time_mean', 'total_time_std']


### 4.2 Comparison: no selection vs correlation filter vs PCA
Same models and cross-validation as in Task 3. PCA keeps 95% of the variance (as in Assignment 1).

In [10]:
from sklearn.decomposition import PCA

# Felles start: preprocessing + feature extraction fra steg 2 og 3
def base_steps():
    return [
        ("capper", IQRCapper()),
        ("scaler", StandardScaler()),
        ("aggregator", TaskAggregator(columns=list(X_train.columns))),
        ("scaler2", StandardScaler()),
    ]

variants = {
    "36 (no selection)": [],
    "28 (correlation filter)": [("selector", CorrelationFilter(threshold=0.9))],
    "PCA (95% variance)": [("pca", PCA(n_components=0.95))],
}

rows = []
for name, model in models.items():
    for label, extra in variants.items():
        pipe = Pipeline(base_steps() + extra + [("model", model)])
        scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["accuracy", "recall"])
        rows.append({
            "Model": name,
            "Features": label,
            "Accuracy": f"{scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}",
            "Recall (P)": f"{scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}",
        })

print(pd.DataFrame(rows).to_string(index=False))

# Hvor mange komponenter trenger PCA?
pca = PCA(n_components=0.95).fit(X_train_fe)
print("\nPCA-komponenter for 95 % varians:", pca.n_components_)

        Model                Features      Accuracy    Recall (P)
          KNN       36 (no selection) 0.928 ± 0.051 0.958 ± 0.034
          KNN 28 (correlation filter) 0.942 ± 0.054 0.958 ± 0.034
          KNN      PCA (95% variance) 0.928 ± 0.051 0.972 ± 0.034
    SVM (RBF)       36 (no selection) 0.914 ± 0.083 0.915 ± 0.083
    SVM (RBF) 28 (correlation filter) 0.899 ± 0.099 0.901 ± 0.107
    SVM (RBF)      PCA (95% variance) 0.914 ± 0.083 0.915 ± 0.083
Decision tree       36 (no selection) 0.885 ± 0.041 0.901 ± 0.073
Decision tree 28 (correlation filter) 0.885 ± 0.041 0.901 ± 0.073
Decision tree      PCA (95% variance) 0.806 ± 0.083 0.803 ± 0.070

PCA-komponenter for 95 % varians: 16
